# EfficientNet-B0 Multi-Class Focal Loss Result Analysis

This notebook analyzes the `EfficientNet-B0` multi-class experiment with `crop + CLAHE + focal loss` for diabetic retinopathy grading on the APTOS 2019 dataset.
The goal is to document the quantitative test results, inspect class-wise behavior, and specifically evaluate whether focal loss improves the rare severe classes.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch.utils.data import DataLoader

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


In [ ]:
project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / "src").exists() and (candidate / "configs").exists():
        project_root = candidate
        break

sys.path.insert(0, str(project_root / "src"))

from dr_detection.config import load_yaml_config, resolve_path
from dr_detection.dataset import FundusImageDataset, default_transforms, load_fundus_records
from dr_detection.train import build_model

config = load_yaml_config(project_root / "configs" / "efficientnet_b0_multiclass_crop_clahe_focal.yaml")
dataset_cfg = config["dataset"]
training_cfg = config["training"]

checkpoint_path = resolve_path(project_root, config["paths"]["model_dir"]) / "best_model.pt"
figures_dir = resolve_path(project_root, config["paths"]["output_dir"])
figures_dir.mkdir(parents=True, exist_ok=True)

class_labels = [0, 1, 2, 3, 4]
class_names = {
    0: "No DR",
    1: "Mild DR",
    2: "Moderate DR",
    3: "Severe DR",
    4: "Proliferative DR",
}

test_records = load_fundus_records(
    csv_path=resolve_path(project_root, dataset_cfg["test_csv_path"]),
    images_dir=resolve_path(project_root, dataset_cfg["test_images_dir"]),
    image_extension=dataset_cfg.get("image_extension", ".png"),
    binary_classification=dataset_cfg.get("binary_classification", True),
)

test_dataset = FundusImageDataset(
    test_records,
    transform=default_transforms(
        image_size=dataset_cfg.get("image_size", 224),
        train=False,
        preprocessing_cfg=dataset_cfg.get("preprocessing"),
    ),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=training_cfg.get("batch_size", 16),
    shuffle=False,
    num_workers=0,
)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

if not checkpoint_path.exists():
    raise FileNotFoundError(
        "Weighted sampler checkpoint not found. Run training first:\n"
        "python3 scripts/train_baseline.py --config configs/efficientnet_b0_multiclass_crop_clahe_focal.yaml\n"
        f"Expected file: {checkpoint_path}"
    )

checkpoint = torch.load(checkpoint_path, map_location=device)
model = build_model(
    model_name=checkpoint["model_name"],
    num_classes=checkpoint["num_classes"],
    pretrained=False,
)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print(f"Checkpoint: {checkpoint_path}")
print(f"Device: {device}")
print(f"Test size: {len(test_dataset)}")


In [ ]:
all_labels = []
all_predictions = []
all_probabilities = []
all_image_paths = []

with torch.no_grad():
    for images, labels, image_paths in test_loader:
        images = images.to(device)
        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        predictions = probabilities.argmax(dim=1)

        all_labels.extend(labels.tolist())
        all_predictions.extend(predictions.cpu().tolist())
        all_probabilities.extend(probabilities.cpu().tolist())
        all_image_paths.extend(image_paths)

predictions_df = pd.DataFrame(
    {
        "image_path": all_image_paths,
        "true_label": all_labels,
        "predicted_label": all_predictions,
    }
)
predictions_df["true_class"] = predictions_df["true_label"].map(class_names)
predictions_df["predicted_class"] = predictions_df["predicted_label"].map(class_names)
predictions_df.to_csv(figures_dir / "table_14_0_multiclass_focal_predictions.csv", index=False)

probability_df = pd.DataFrame(all_probabilities, columns=[f"prob_class_{label}" for label in class_labels])
predictions_df = pd.concat([predictions_df, probability_df], axis=1)
print(f"Collected predictions for {len(predictions_df)} test images.")
predictions_df.head()


**Table 14.1. Final test-set metrics for the EfficientNet-B0 multi-class focal loss experiment**


In [ ]:
probability_array = np.asarray(all_probabilities, dtype=np.float32)

overall_metrics_df = pd.DataFrame(
    {
        "metric": [
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "f1_weighted",
            "qwk",
            "roc_auc_ovr_macro",
            "roc_auc_ovr_weighted",
        ],
        "value": [
            accuracy_score(all_labels, all_predictions),
            precision_recall_fscore_support(all_labels, all_predictions, labels=class_labels, average="macro", zero_division=0)[0],
            precision_recall_fscore_support(all_labels, all_predictions, labels=class_labels, average="macro", zero_division=0)[1],
            precision_recall_fscore_support(all_labels, all_predictions, labels=class_labels, average="macro", zero_division=0)[2],
            precision_recall_fscore_support(all_labels, all_predictions, labels=class_labels, average="weighted", zero_division=0)[2],
            cohen_kappa_score(all_labels, all_predictions, weights="quadratic"),
            roc_auc_score(all_labels, probability_array, labels=class_labels, multi_class="ovr", average="macro"),
            roc_auc_score(all_labels, probability_array, labels=class_labels, multi_class="ovr", average="weighted"),
        ],
    }
)
overall_metrics_df["value"] = overall_metrics_df["value"].round(6)
overall_metrics_df.to_csv(figures_dir / "table_14_1_multiclass_focal_metrics.csv", index=False)
overall_metrics_df


**Table 14.2. Per-class precision, recall, F1-score, and support for the EfficientNet-B0 focal loss experiment**


In [ ]:
report = classification_report(
    all_labels,
    all_predictions,
    labels=class_labels,
    target_names=[class_names[label] for label in class_labels],
    output_dict=True,
    zero_division=0,
)

per_class_metrics_df = pd.DataFrame(
    [
        {
            "class_id": label,
            "class_name": class_names[label],
            "precision": report[class_names[label]]["precision"],
            "recall": report[class_names[label]]["recall"],
            "f1_score": report[class_names[label]]["f1-score"],
            "support": int(report[class_names[label]]["support"]),
        }
        for label in class_labels
    ]
)
per_class_metrics_df[["precision", "recall", "f1_score"]] = per_class_metrics_df[["precision", "recall", "f1_score"]].round(4)
per_class_metrics_df.to_csv(figures_dir / "table_14_2_multiclass_focal_per_class_metrics.csv", index=False)
per_class_metrics_df


**Table 14.3. Confusion-matrix counts for the EfficientNet-B0 focal loss experiment**


In [ ]:
cm = confusion_matrix(all_labels, all_predictions, labels=class_labels)
cm_counts_df = pd.DataFrame(
    cm,
    index=[f"True: {class_names[label]}" for label in class_labels],
    columns=[f"Pred: {class_names[label]}" for label in class_labels],
)
cm_counts_df.to_csv(figures_dir / "table_14_3_multiclass_focal_confusion_matrix_counts.csv")
cm_counts_df


**Figure 14.1. Confusion matrix for the EfficientNet-B0 focal loss experiment on the APTOS 2019 test split**


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=[class_names[label] for label in class_labels],
    yticklabels=[class_names[label] for label in class_labels],
    ax=ax,
)
ax.set_title("Confusion Matrix for EfficientNet-B0 Multi-Class DR Grading (Focal Loss)")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
fig.savefig(figures_dir / "figure_14_1_multiclass_focal_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()


**Figure 14.2. Row-normalized confusion matrix for the EfficientNet-B0 focal loss experiment**


In [ ]:
cm_normalized = cm.astype(np.float32) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    cbar=True,
    xticklabels=[class_names[label] for label in class_labels],
    yticklabels=[class_names[label] for label in class_labels],
    ax=ax,
)
ax.set_title("Row-Normalized Confusion Matrix for EfficientNet-B0 Multi-Class DR Grading (Focal Loss) (Focal Loss)")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
fig.savefig(figures_dir / "figure_14_2_multiclass_focal_confusion_matrix_normalized.png", dpi=300, bbox_inches="tight")
plt.show()


**Figure 14.3. Per-class precision, recall, and F1-score for the EfficientNet-B0 focal loss experiment**


In [ ]:
plot_df = per_class_metrics_df.melt(
    id_vars=["class_id", "class_name", "support"],
    value_vars=["precision", "recall", "f1_score"],
    var_name="metric",
    value_name="value",
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=plot_df, x="class_name", y="value", hue="metric", ax=ax)
ax.set_title("Per-Class Metrics for EfficientNet-B0 Multi-Class DR Grading (Focal Loss)")
ax.set_xlabel("Class")
ax.set_ylabel("Value")
ax.set_ylim(0, 1.02)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig.savefig(figures_dir / "figure_14_3_multiclass_focal_per_class_metrics.png", dpi=300, bbox_inches="tight")
plt.show()


**Table 14.4. Rare-class recall and F1-score comparison between the previous best preprocessing setup and the focal loss experiment**


In [ ]:
previous_per_class_f1_df = pd.read_csv(project_root / "outputs" / "multiclass_comparison" / "table_12_2_multiclass_per_class_f1.csv")
previous_rare_recall_df = pd.read_csv(project_root / "outputs" / "multiclass_comparison" / "table_12_3_multiclass_rare_class_recall.csv")

current_rare_df = per_class_metrics_df[
    per_class_metrics_df["class_name"].isin(["Severe DR", "Proliferative DR"])
][["class_name", "recall", "f1_score"]].copy()
current_rare_df = current_rare_df.rename(
    columns={
        "recall": "Focal Loss Recall",
        "f1_score": "Focal Loss F1",
    }
)

previous_best_df = previous_rare_recall_df[["class_name", "Crop + CLAHE"]].rename(
    columns={"Crop + CLAHE": "Crop + CLAHE Recall"}
)
previous_best_f1_df = previous_per_class_f1_df[["class_name", "Crop + CLAHE"]].rename(
    columns={"Crop + CLAHE": "Crop + CLAHE F1"}
)

rare_class_comparison_df = previous_best_df.merge(previous_best_f1_df, on="class_name", how="inner")
rare_class_comparison_df = rare_class_comparison_df.merge(current_rare_df, on="class_name", how="inner")
rare_class_comparison_df = rare_class_comparison_df[[
    "class_name",
    "Crop + CLAHE Recall",
    "Focal Loss Recall",
    "Crop + CLAHE F1",
    "Focal Loss F1",
]].round(4)

rare_class_comparison_df.to_csv(
    figures_dir / "table_14_4_focal_rare_class_comparison.csv",
    index=False,
)
rare_class_comparison_df


**Figure 14.4. Rare-class recall and F1-score comparison for `Severe DR` and `Proliferative DR` after introducing focal loss**


In [ ]:
recall_plot_df = rare_class_comparison_df[["class_name", "Crop + CLAHE Recall", "Focal Loss Recall"]].melt(
    id_vars="class_name",
    var_name="experiment",
    value_name="value",
)
recall_plot_df["experiment"] = recall_plot_df["experiment"].str.replace(" Recall", "", regex=False)

f1_plot_df = rare_class_comparison_df[["class_name", "Crop + CLAHE F1", "Focal Loss F1"]].melt(
    id_vars="class_name",
    var_name="experiment",
    value_name="value",
)
f1_plot_df["experiment"] = f1_plot_df["experiment"].str.replace(" F1", "", regex=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.barplot(data=recall_plot_df, x="class_name", y="value", hue="experiment", ax=axes[0])
axes[0].set_title("Rare-Class Recall Comparison")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Recall")
axes[0].set_ylim(0, 1.02)
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=f1_plot_df, x="class_name", y="value", hue="experiment", ax=axes[1])
axes[1].set_title("Rare-Class F1 Comparison")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("F1-score")
axes[1].set_ylim(0, 1.02)
axes[1].tick_params(axis="x", rotation=15)

handles, labels = axes[1].get_legend_handles_labels()
axes[0].legend_.remove()
axes[1].legend(handles=handles, labels=labels, title="Experiment")
plt.tight_layout()
fig.savefig(
    figures_dir / "figure_14_4_focal_rare_class_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()
